

###### 07_mlflow_tracking

###### Purpose
Validate the MLflow experiment, inspect logged runs and artifacts, verify the best model, and understand the contents of the saved MLflow model before registration.

###### Technologies Used

- Databricks

- MLflow

- Scikit-learn

- Delta Lake

###### Input

- MLflow experiment path

- Logged MLflow runs

- best_model_info Delta table

- ```Logged MLflow model artifact (runs:/<run_id>/model)```

######  Output

- Verified MLflow experiment

- Displayed experiment metadata

- Compared all model runs

- Validated best model information

- Inspected logged model artifacts

- Verified model signature and pipeline contents

######  Architecture

```text

Current User
      ↓
Build Experiment Path
      ↓
Load MLflow Experiment
      ↓
Display Experiment Metadata
      ↓
Search Runs
      ↓
Compare Metrics
      ↓
Validate best_model_info
      ↓
Read Selected Run ID
        ↓
Build runs:/ Model URI
        ↓
Load Logged Scikit-learn Pipeline
        ↓
Inspect Preprocessor and Classifier
        ↓
Inspect Model Signature

```

###### Section 0 : Call project config notebook

In [0]:
%run ./00_project_config

###### Section 1: Load and Validate the MLflow Experiment

In [0]:
import mlflow

user_name = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)

experiment_path = f"/Users/{user_name}/telco_churn_experiment"
experiment = mlflow.get_experiment_by_name(experiment_path)

if experiment is None:
    raise ValueError(f"Experiment not found: {experiment_path}")

print("Experiment loaded successfully.")
print("Experiment ID:", experiment.experiment_id)

###### Section 2 : Compare Logged Runs

In [0]:
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.roc_auc DESC"]
)

if runs_df.empty:
    raise ValueError("No MLflow runs were found in the experiment.")

display(runs_df)

###### Section 3 : Validate the Selected Model

In [0]:
best_model_row = spark.table(BEST_MODEL_TABLE).first()

if best_model_row is None:
    raise ValueError("Best-model metadata table is empty.")

best_run_id = best_model_row["best_run_id"]
top_run_id = runs_df.iloc[0]["run_id"]

assert best_run_id == top_run_id

display(spark.table(BEST_MODEL_TABLE))

###### Section 4 : Load and Inspect the Logged Model

In [0]:
model_uri = f"runs:/{best_run_id}/model"

loaded_model = mlflow.sklearn.load_model(model_uri)

print("Loaded object type:", type(loaded_model))
print("Pipeline steps:", loaded_model.named_steps.keys())

preprocessor = loaded_model.named_steps["preprocessor"]
classifier = loaded_model.named_steps["model"]

print("Preprocessor:")
print(preprocessor)

print("Classifier:")
print(classifier)

###### Section 5: Inspect Model Metadata and Signature

In [0]:
model_info = mlflow.models.get_model_info(model_uri)

print("Model URI:", model_info.model_uri)
print("Model signature:")
print(model_info.signature)


###### Notebook Summary

- Validated the MLflow experiment path.

- Displayed experiment metadata.

- Compared all logged model runs.

- Verified the selected best model.

- Inspected the serialized sklearn pipeline.

- Explored classifier.

- Verified the model signature.

######  Key Learnings

-  model.pkl stores the serialized Scikit-learn pipeline containing the preprocessor and classification model.

-  The MLmodel file stores MLflow model metadata, including supported model flavors and signature information.

-  A model signature defines the expected model inputs and outputs.

-  An input example documents representative input data and helps validate the model before serving.

-  MLflow model artifacts contain the files needed to load and reproduce the trained model in a compatible environment.

-  Model registration creates a governed model version from an existing logged model artifact; it does not retrain the model.

###### Notebook Conclusion

- In this notebook, we inspected the MLflow experiment, compared its logged runs, and verified that the selected model corresponds to the highest-ranked ROC-AUC run.

- Loaded the selected Scikit-learn pipeline directly from its MLflow run and inspected its preprocessing steps, classification model, and input/output signature.

- The validated model URI will be used in the next notebook to create a versioned model in Unity Catalog Model Registry.


###### Next Notebook

08_model_registry

- Register the best Telco churn classification model from MLflow into Unity Catalog Model Registry.